# Peptide Biophysical Property Calculator
### FASTA → Biophysical Features → CSV (Google Colab Ready)

This notebook computes a comprehensive set of **biophysical properties** for each peptide/protein sequence in a FASTA file — useful for antimicrobial peptide (AMP), cell-penetrating peptide (CPP), and general therapeutic peptide characterization.

**Pipeline:**
1. Install dependencies
2. Upload / load a FASTA file
3. Parse sequences
4. Compute physicochemical, compositional, hydrophobicity, structural, and complexity features
5. Assemble a single results table: `Sequence, MW, Feature1, Feature2, ...`
6. Export to CSV (and download automatically in Colab)

> Run the cells in order (Runtime → Run all).


## Step 1 — Install Required Libraries

In [ ]:
!pip -q install biopython peptides pandas matplotlib numpy
print("Libraries installed.")

## Step 2 — Upload Your FASTA File

Run the cell below and select your `.fasta` / `.fa` file when prompted.
(If you're not on Colab, just set `FASTA_PATH` manually and skip the upload widget.)

In [ ]:
import os

FASTA_PATH = None  # will be set automatically after upload

try:
    from google.colab import files
    uploaded = files.upload()
    FASTA_PATH = list(uploaded.keys())[0]
    print(f"Uploaded file: {FASTA_PATH}")
except ImportError:
    # Not running in Colab -- set the path manually here
    FASTA_PATH = "peptides.fasta"
    print(f"Not running in Colab. Using manual path: {FASTA_PATH}")

assert FASTA_PATH is not None and os.path.exists(FASTA_PATH), \
    "FASTA file not found. Upload a file or set FASTA_PATH manually."


## Step 3 — Parse the FASTA File

Some FASTA files include comment lines (starting with `;` or `#`) before the
first `>` header, which BioPython's strict `fasta` parser rejects. The cell
below cleans those out automatically, then parses normally — no manual
file editing required.

In [ ]:
from Bio import SeqIO
import io

# Some FASTA files include comment lines (starting with ';' or '#') before the
# first '>' header, which BioPython's strict 'fasta' parser rejects.
# Strip any such leading lines out automatically, then parse normally.
with open(FASTA_PATH) as fh:
    lines = fh.readlines()

first_header = next((i for i, line in enumerate(lines) if line.startswith(">")), None)
assert first_header is not None, "No FASTA header ('>') found in the file."
cleaned = "".join(lines[first_header:])

records = list(SeqIO.parse(io.StringIO(cleaned), "fasta"))

# Drop any empty sequences, then relabel with simple sequential IDs
# (keeping the original FASTA header safe in .description for reference)
records = [r for r in records if len(r.seq) > 0]
for i, r in enumerate(records):
    r.description = r.id  # preserve original header text
    r.id = f"Pep_{i+1}"

print(f"Loaded {len(records)} sequences.\n")

for r in records[:5]:
    print(r.id, len(r.seq), str(r.seq)[:20], "...")

## Step 4 — Helper Functions

Custom functions for properties not directly provided by BioPython / `peptides`:
Shannon entropy, hydrophobic residue ratio, and residue-class counts.

In [ ]:
import math

HYDROPHOBIC_RESIDUES = set("AILMFWVY")
POLAR_RESIDUES       = set("STNQCY")
NONPOLAR_RESIDUES    = set("AVLIMFWPG")
ACIDIC_RESIDUES      = set("DE")
BASIC_RESIDUES       = set("KRH")
TINY_RESIDUES        = set("AGCS")
SMALL_RESIDUES       = set("AGCSPNDT")
SULFUR_RESIDUES      = set("CM")


def shannon_entropy(sequence: str) -> float:
    """Shannon entropy (bits) of the amino acid distribution in a sequence."""
    freq = {}
    for aa in sequence:
        freq[aa] = freq.get(aa, 0) + 1
    n = len(sequence)
    h = 0.0
    for f in freq.values():
        p = f / n
        h -= p * math.log2(p)
    return h


def residue_ratio(sequence: str, residue_set: set) -> float:
    """Percentage of residues in `sequence` that belong to `residue_set`."""
    if len(sequence) == 0:
        return 0.0
    return sum(aa in residue_set for aa in sequence) / len(sequence) * 100


def residue_count(sequence: str, residue_set: set) -> int:
    """Raw count of residues in `sequence` that belong to `residue_set`."""
    return sum(aa in residue_set for aa in sequence)


## Step 5 — Compute Biophysical Features

For every sequence we compute:

| Category | Features |
|---|---|
| Basic physicochemical | Molecular weight, length, pI, net charge |
| Hydrophobicity | GRAVY, hydrophobic moment, hydrophobic ratio |
| Structural | Instability index, aliphatic index, helix/turn/sheet fraction |
| Membrane interaction | Boman index |
| Composition | Aromaticity, positive/negative/polar/nonpolar/tiny/small residue % |
| Complexity | Shannon entropy |
| Specific residues | Proline, Glycine, Cysteine content (%) |


In [ ]:
import pandas as pd
import peptides
from Bio.SeqUtils.ProtParam import ProteinAnalysis

results = []

for record in records:
    seq = str(record.seq).upper().replace("*", "")  # strip stop codons if present
    if len(seq) == 0:
        continue

    try:
        X = ProteinAnalysis(seq)
        pep = peptides.Peptide(seq)
        helix, turn, sheet = X.secondary_structure_fraction()

        row = {
            "ID": record.id,
            "Sequence": seq,
            "MW": round(X.molecular_weight(), 3),
            "Length": len(seq),
            "pI": round(X.isoelectric_point(), 3),
            "Charge_pH7": round(pep.charge(pH=7.0), 3),
            "GRAVY": round(X.gravy(), 4),
            "Aromaticity": round(X.aromaticity(), 4),
            "InstabilityIndex": round(X.instability_index(), 3),
            "AliphaticIndex": round(pep.aliphatic_index(), 3),
            "BomanIndex": round(pep.boman(), 3),
            "HydrophobicMoment": round(pep.hydrophobic_moment(), 4),
            "HydrophobicRatio": round(residue_ratio(seq, HYDROPHOBIC_RESIDUES), 2),
            "HelixFraction": round(helix, 4),
            "TurnFraction": round(turn, 4),
            "SheetFraction": round(sheet, 4),
            "ShannonEntropy": round(shannon_entropy(seq), 4),
            "PositiveResidues_pct": round(residue_ratio(seq, BASIC_RESIDUES), 2),
            "NegativeResidues_pct": round(residue_ratio(seq, ACIDIC_RESIDUES), 2),
            "PolarResidues_pct": round(residue_ratio(seq, POLAR_RESIDUES), 2),
            "NonpolarResidues_pct": round(residue_ratio(seq, NONPOLAR_RESIDUES), 2),
            "TinyResidues_pct": round(residue_ratio(seq, TINY_RESIDUES), 2),
            "SmallResidues_pct": round(residue_ratio(seq, SMALL_RESIDUES), 2),
            "SulfurResidues_pct": round(residue_ratio(seq, SULFUR_RESIDUES), 2),
            "Proline_pct": round(residue_ratio(seq, {"P"}), 2),
            "Glycine_pct": round(residue_ratio(seq, {"G"}), 2),
            "Cysteine_pct": round(residue_ratio(seq, {"C"}), 2),
        }
        results.append(row)

    except Exception as e:
        print(f"Skipped {record.id}: {e}")

print(f"Computed features for {len(results)} / {len(records)} sequences.")


## Step 6 — Assemble & Order the Results Table

Final column order:
`ID, Sequence, MW, Feature1, Feature2, Feature3, ...`

In [ ]:
df = pd.DataFrame(results)

# Enforce the requested column order: ID, Sequence, MW, then all remaining features
ordered_cols = ["ID", "Sequence", "MW"] + [c for c in df.columns if c not in ("ID", "Sequence", "MW")]
df = df[ordered_cols]

print(f"Final table shape: {df.shape[0]} sequences x {df.shape[1]} columns")
df.head()


## Step 7 — Save & Download the CSV

In [ ]:
OUTPUT_CSV = "Peptide_Biophysical_Properties.csv"
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")

try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except ImportError:
    print("Not on Colab -- file saved locally, no auto-download.")


## Step 8 — Preview Full Results

In [ ]:
pd.set_option('display.max_columns', None)
df

---
## Optional Extras

The cells below are **not required** for the main CSV output, but are useful add-ons: amino acid composition, dipeptide composition, and net charge vs. pH.

### Optional A — Amino Acid Composition Table (per sequence)

In [ ]:
aa_rows = []

for record in records:
    seq = str(record.seq).upper().replace("*", "")

    if len(seq) == 0:
        continue

    X = ProteinAnalysis(seq)

    comp = X.amino_acids_percent

    comp_row = {
        "ID": record.id,
        "Sequence": seq
    }

    comp_row.update({f"pct_{aa}": round(v, 2) for aa, v in comp.items()})

    aa_rows.append(comp_row)

aa_df = pd.DataFrame(aa_rows)

aa_df.to_csv("Peptide_AminoAcid_Composition.csv", index=False)

aa_df.head()

In [ ]:
import matplotlib.pyplot as plt

aa_cols = [c for c in aa_df.columns if c.startswith("pct_")]

mean_comp = aa_df[aa_cols].mean()

plt.figure(figsize=(12,5))

plt.bar(mean_comp.index.str.replace("pct_",""), mean_comp.values)

plt.xlabel("Amino Acid")

plt.ylabel("Average Composition (%)")

plt.title("Average Amino Acid Composition")

plt.tight_layout()

plt.show()

### Optional B — Dipeptide Composition (per sequence)

In [ ]:
from collections import Counter

dipeptide_rows = []
for record in records:
    seq = str(record.seq).upper().replace("*", "")
    if len(seq) < 2:
        continue
    dipeptides = [seq[i:i+2] for i in range(len(seq) - 1)]
    counts = Counter(dipeptides)
    row = {"ID": record.id, "Sequence": seq}
    row.update(counts)
    dipeptide_rows.append(row)

dipeptide_df = pd.DataFrame(dipeptide_rows).fillna(0)
dipeptide_df.to_csv("Peptide_Dipeptide_Composition.csv", index=False)
dipeptide_df.head()


In [ ]:
import matplotlib.pyplot as plt

# Remove metadata columns
feature_cols = [c for c in dipeptide_df.columns if c not in ["ID", "Sequence"]]

# Mean occurrence of each dipeptide
mean_counts = dipeptide_df[feature_cols].mean()

# Top 20
top20 = mean_counts.sort_values(ascending=False).head(20)

plt.figure(figsize=(12,6))

plt.bar(top20.index, top20.values)

plt.xticks(rotation=90)

plt.xlabel("Dipeptide")

plt.ylabel("Average Frequency")

plt.title("Top 20 Most Frequent Dipeptides")

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

top30 = mean_counts.sort_values(ascending=False).head(30).index

plt.figure(figsize=(14,8))

plt.imshow(dipeptide_df[top30], aspect='auto')

plt.colorbar(label="Count")

plt.xticks(
    range(len(top30)),
    top30,
    rotation=90
)

plt.yticks([])

plt.xlabel("Dipeptides")

plt.ylabel("Peptides")

plt.title("Heatmap of Top 30 Dipeptides")

plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

AA = "ACDEFGHIKLMNPQRSTVWY"

matrix = np.zeros((20,20))

for i,a1 in enumerate(AA):
    for j,a2 in enumerate(AA):

        dp = a1+a2

        if dp in mean_counts:
            matrix[i,j] = mean_counts[dp]

plt.figure(figsize=(10,8))

plt.imshow(matrix)

plt.colorbar(label="Average Frequency")

plt.xticks(range(20), list(AA))

plt.yticks(range(20), list(AA))

plt.xlabel("Second Residue")

plt.ylabel("First Residue")

plt.title("Average Dipeptide Composition Matrix")

plt.tight_layout()

plt.show()

### Optional C — Net Charge vs. pH (plot per sequence)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ph_range = np.arange(2, 12.5, 0.5)

plt.figure(figsize=(8, 5))
for record in records:
    seq = str(record.seq).upper().replace("*", "")
    if len(seq) == 0:
        continue
    pep = peptides.Peptide(seq)
    charges = [pep.charge(pH=p) for p in ph_range]
    plt.plot(ph_range, charges, marker="o", markersize=3, label=record.id)

plt.axhline(0, color="grey", linewidth=0.8, linestyle="--")
plt.xlabel("pH")
plt.ylabel("Net Charge")
plt.title("Net Charge vs. pH")
if len(records) <= 15:
    plt.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.savefig("Charge_vs_pH.png", dpi=150)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ph_range = np.arange(2, 12.5, 0.5)

charge_matrix = []

for record in records:

    seq = str(record.seq).replace("*","").upper()

    if len(seq)==0:
        continue

    pep = peptides.Peptide(seq)

    charge_matrix.append(
        [pep.charge(pH=p) for p in ph_range]
    )

charge_matrix = np.array(charge_matrix)

mean_charge = charge_matrix.mean(axis=0)
std_charge  = charge_matrix.std(axis=0)

plt.figure(figsize=(8,6))

plt.plot(ph_range, mean_charge, linewidth=3)

plt.fill_between(
    ph_range,
    mean_charge-std_charge,
    mean_charge+std_charge,
    alpha=0.3
)

plt.axhline(0,color='black',ls='--')

plt.xlabel("pH")

plt.ylabel("Net Charge")

plt.title("Average Net Charge vs pH")

plt.tight_layout()

plt.show()